# **Installing required modules and packages**

In [2]:
# !pip install tensorflow opencv-python mediapipe scikit-learn matplotlib numpy pandas
!pip show mediapipe

Name: mediapipe
Version: 0.10.21
Summary: MediaPipe is the simplest way for researchers and developers to build world-class ML solutions and applications for mobile, edge, cloud and the web.
Home-page: https://github.com/google/mediapipe
Author: The MediaPipe Authors
Author-email: mediapipe@google.com
License: Apache 2.0
Location: /home/sanjib/Desktop/backend/mp-env/lib/python3.12/site-packages
Requires: absl-py, attrs, flatbuffers, jax, jaxlib, matplotlib, numpy, opencv-contrib-python, protobuf, sentencepiece, sounddevice
Required-by: 


In [ ]:
import mediapipe as mp
# import mediapipe.python.solutions.holistic as mp_holistic
mp_holistic = mp.solutions.holistic

ho = mp_holistic.Holistic()
print(ho)

I0000 00:00:1751173154.065608    8132 gl_context_egl.cc:85] Successfully initialized EGL. Major : 1 Minor: 5
I0000 00:00:1751173154.071384   10616 gl_context.cc:369] GL version: 3.2 (OpenGL ES 3.2 Mesa 24.1.4-arch1.2), renderer: Mesa Intel(R) Graphics (ADL GT2)


W0000 00:00:1751173154.195731   10608 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1751173154.243893   10605 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1751173154.247984   10605 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1751173154.249620   10611 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1751173154.249656   10606 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1751173154.266167   10604 inference_feedback_manager.cc:114] Feedback manager 

# **Importing packages**

In [6]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
import mediapipe as mp
import tensorflow as tf

# **Capturing video and keypoints using mediapipe holistic**

In [7]:
mp_holistic = mp.solutions.holistic
mp_drawing = mp.solutions.drawing_utils

filtered_hand = list(range(21))
filtered_pose = [11, 12, 13, 14, 15, 16]
filtered_face = [0, 4, 7, 8, 10, 13, 14, 17, 21, 33, 37, 39, 40, 46, 52, 53, 54, 55, 58,
                 61, 63, 65, 66, 67, 70, 78, 80, 81, 82, 84, 87, 88, 91, 93, 95, 103, 105,
                 107, 109, 127, 132, 133, 136, 144, 145, 146, 148, 149, 150, 152, 153, 154,
                 155, 157, 158, 159, 160, 161, 162, 163, 172, 173, 176, 178, 181, 185, 191, 215,
                 234, 246, 249, 251, 263, 267, 269, 270, 276, 282, 283, 284, 285, 288, 291,
                 293, 295, 296, 297, 300, 308, 310, 311, 312, 314, 317, 318, 321, 323, 324,
                 332, 334, 336, 338, 356, 361, 362, 365, 373, 374, 375, 377, 378, 379, 380,
                 381, 382, 384, 385, 386, 387, 388, 389, 390, 397, 398, 400, 402, 405, 409,
                 415, 435, 454, 466]

def filter_landmarks(landmarks, indices):
    """Return a list of only selected landmarks."""
    if landmarks is None:
        return None
    from mediapipe.framework.formats import landmark_pb2
    filtered = landmark_pb2.NormalizedLandmarkList()
    for i in indices:
        if i < len(landmarks.landmark):
            filtered.landmark.append(landmarks.landmark[i])
    return filtered

In [5]:
def mediapipe_detection(image, model):
  image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
  image.flags.writeable = False
  results = model.process(image)
  image.flags.writeable = True
  image = cv2.cvtColor(image, cv2.COLOR_RGB2BGR)
  return image, results

In [6]:
def draw_landmarks(image, results):
    mp_drawing.draw_landmarks(
        image, filter_landmarks(results.face_landmarks, filtered_face))
    mp_drawing.draw_landmarks(
        image, filter_landmarks(results.pose_landmarks, filtered_pose))
    mp_drawing.draw_landmarks(
        image, filter_landmarks(results.left_hand_landmarks, filtered_hand), mp_holistic.HAND_CONNECTIONS)
    mp_drawing.draw_landmarks(
        image, filter_landmarks(results.right_hand_landmarks, filtered_hand), mp_holistic.HAND_CONNECTIONS)


In [7]:
def draw_styled_landmarks(image, results):
    # Face
    mp_drawing.draw_landmarks(
        image, filter_landmarks(results.face_landmarks, filtered_face), connections = None,
        landmark_drawing_spec=mp_drawing.DrawingSpec(color=(0, 255, 0), thickness=1, circle_radius=1)
    )
    # Pose
    mp_drawing.draw_landmarks(
        image, filter_landmarks(results.pose_landmarks, filtered_pose), connections = None,
        landmark_drawing_spec=mp_drawing.DrawingSpec(color=(80, 22, 10), thickness=2, circle_radius=1),
    )
    # Left hand
    mp_drawing.draw_landmarks(
        image, filter_landmarks(results.left_hand_landmarks, filtered_hand), mp_holistic.HAND_CONNECTIONS,
        mp_drawing.DrawingSpec(color=(121,22,76), thickness=2, circle_radius=2),
        mp_drawing.DrawingSpec(color=(121,44,250), thickness=2, circle_radius=1)
    )
    # Right hand
    mp_drawing.draw_landmarks(
        image, filter_landmarks(results.right_hand_landmarks, filtered_hand), mp_holistic.HAND_CONNECTIONS,
        mp_drawing.DrawingSpec(color=(245,117,66), thickness=2, circle_radius=2),
        mp_drawing.DrawingSpec(color=(245,66,230), thickness=2, circle_radius=1)
    )


**Below code is for checking if hand is detected**

In [25]:
cap = cv2.VideoCapture(0)
with mp_holistic.Holistic(min_detection_confidence=0.5, min_tracking_confidence=0.5) as holistic_model:
  while cap.isOpened():
      ret, frame = cap.read()

      # make detections
      image, results = mediapipe_detection(frame, holistic_model)
      # print(results)

      #draw landmarks
      draw_landmarks(image, results)

      cv2.imshow('OpenCV Feed', image)

      if cv2.waitKey(10) & 0xFF == ord('q'):
          break

  cap.release()
  cv2.destroyAllWindows()

In [23]:
# with mp_holistic.Holistic(min_detection_confidence=0.5, min_tracking_confidence=0.5) as holistic_model:
#   image_data = cv2.cvtColor(cv2.imread("C:/Users/Acer/Pictures/Screenshots/Screenshot 2025-06-27 200915.png"), cv2.COLOR_BGR2RGB)
#   image, results = mediapipe_detection(image_data, holistic_model)
#   draw_styled_landmarks(image, results)
#   plt.imshow(image)
#   plt.show()

# **Extract Keypoint values**

In [22]:
# print(results.face_landmarks.landmark[0])
# print(results.pose_landmarks.landmark[0])
# print(results.left_hand_landmarks.landmark[0])
# print(results.right_hand_landmarks.landmark[0])

In [12]:
# pose = []

# for res in results.pose_landmarks.landmark:
#   test = np.array([res.x, res.y, res.z, res.visibility]).flatten()
#   pose.append(test)
# pose
# len(pose)
# def extract_keypoints(results):
#   pose = np.array([[res.x, res.y, res.z, res.visibility] for res in results.pose_landmarks.landmark]).flatten() if results.pose_landmarks else np.zeros(132)
#   face = np.array([[res.x, res.y, res.z] for res in results.face_landmarks.landmark]).flatten() if results.face_landmarks else np.zeros(1404)
#   lh = np.array([[res.x, res.y, res.z] for res in results.left_hand_landmarks.landmark]).flatten() if results.left_hand_landmarks else np.zeros(21*3)
#   rh = np.array([[res.x, res.y, res.z] for res in results.right_hand_landmarks.landmark]).flatten() if results.right_hand_landmarks else np.zeros(21*3)
#   return np.concatenate([pose, face, lh, rh])

def extract_keypoints(results):
    # Pose
    if results.pose_landmarks:
        pose = [
            [res.x, res.y, res.z, res.visibility]
            for i, res in enumerate(results.pose_landmarks.landmark)
            if i in filtered_pose
        ]
        pose = np.array(pose).flatten()
    else:
        pose = np.zeros(len(filtered_pose) * 4)

    # Face
    if results.face_landmarks:
        face = [
            [res.x, res.y, res.z]
            for i, res in enumerate(results.face_landmarks.landmark)
            if i in filtered_face
        ]
        face = np.array(face).flatten()
    else:
        face = np.zeros(len(filtered_face) * 3)

    # Left hand
    if results.left_hand_landmarks:
        lh = [
            [res.x, res.y, res.z]
            for i, res in enumerate(results.left_hand_landmarks.landmark)
            if i in filtered_hand
        ]
        lh = np.array(lh).flatten()
    else:
        lh = np.zeros(len(filtered_hand) * 3)

    # Right hand
    if results.right_hand_landmarks:
        rh = [
            [res.x, res.y, res.z]
            for i, res in enumerate(results.right_hand_landmarks.landmark)
            if i in filtered_hand
        ]
        rh = np.array(rh).flatten()
    else:
        rh = np.zeros(len(filtered_hand) * 3)

    return np.concatenate([pose, face, lh, rh])


In [ ]:
# result_test = extract_keypoints(results)
# print(result_test)
# np.save('name', result_test)
# np.load('/content/name.npy')

# **Folders for collecting videos**
**For our project, we will get video from dataset**

In [8]:
import os
import numpy as np
DATA_PATH = os.path.join('../../data') # data folder should be inside backend
os.makedirs(DATA_PATH, exist_ok=True)
# say
# hello
# thankyou
# please
# sorry
# shhh
# help
# text
# pay attention
# look / see / watch
# forget
# understand
# love it
# i
# you
# man
# woman
# deaf
# name
# sad
# fine
# same

# yesma tmro word anusar change garni hai (replace with your own words)
# actions = np.array(['go', 'come', 'walk', 'give', 'cook', 'clean', 'finish', 'work', 'sleep', 'play', 'draw', 'write', 'learn', 'know', 'ask', 'test', 'house', 'room', 'bathroom', 'car']) # <---------
actions = np.array( [
    "always", "ask", "bathroom", "bird", "black", "blue", "book", "brown", "busy", "buy",
    "candy", "car", "cat", "clean", "come", "cook", "deaf", "draw", "drink", "eat",
    "fine", "finish", "forget", "give", "go", "good", "green", "happy", "hello", "help",
    "house", "how", "hungry", "i", "icecream", "know", "learn", "like", "love_it", "man",
    "milk", "more", "name", "never", "no", "not", "pay_attention", "play", "please", "red",
    "right", "room", "sad", "same", "say", "see", "shhh", "sleep", "sorry", "test", "text",
    "thankyou", "time", "today", "tomorrow", "understand", "walk", "want", "water", "what",
    "where", "white", "who", "woman", "work", "write", "wrong", "yes", "yesterday", "you"
]) # <---------

no_sequences = 20

sequence_length = 30 # 25

start_folder = 0
print(str(DATA_PATH))


../../data


In [48]:
for action in actions:
    action_path = os.path.join(DATA_PATH, action)
    if not os.path.exists(action_path):
        os.makedirs(action_path)
        print(f"Created directory: {action_path}")
    else:
        print(f"Directory already exists: {action_path}")

Created directory: ../../data\who
Created directory: ../../data\what
Created directory: ../../data\where
Created directory: ../../data\how


In [49]:
for action in actions:
    action_dir = os.path.join(DATA_PATH, action)
    dirs = [d for d in os.listdir(action_dir) if d.isdigit()]
    if len(dirs) > 0:
        dirmax = np.max(np.array(dirs).astype(int))
    else:
        dirmax = -1
    for sequence in range(1, no_sequences+1):
        try:
            os.makedirs(os.path.join(DATA_PATH, action, str(dirmax+sequence)))
        except:
            pass


# **Collecting keypoints for training and testing**

In [50]:
cap = cv2.VideoCapture(0)
with mp_holistic.Holistic(min_detection_confidence=0.5, min_tracking_confidence=0.5) as holistic:
    stop = False
    for action in actions:
        if stop:
            break
        for sequence in range(start_folder, start_folder+no_sequences):
            if stop:
                break
            for frame_num in range(sequence_length):
                ret, frame = cap.read()
                frame = cv2.flip(frame, 1)  # Flip the frame horizontally
                if not ret:
                    print("Failed to grab frame.")
                    stop = True
                    break

                image, results = mediapipe_detection(frame, holistic)

                draw_styled_landmarks(image, results)

                if frame_num == 0:
                    cv2.putText(image, 'STARTING COLLECTION', (120,200),
                               cv2.FONT_HERSHEY_SIMPLEX, 1, (0,255, 0), 3, cv2.LINE_AA)
                    cv2.putText(image, 'Collecting frames for {} Video No: {}'.format(action, sequence), (15,20),
                               cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 0, 0), 1, cv2.LINE_AA)
                    cv2.imshow('OpenCV Feed', image)
                    cv2.waitKey(2000)
                else:
                    cv2.putText(image, 'Collecting frames for {} Video No: {} frame {}'.format(action, sequence, frame_num), (15,20),
                               cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 0, 0), 1, cv2.LINE_AA)
                    cv2.imshow('OpenCV Feed', image)

                keypoints = extract_keypoints(results)
                npy_dir = os.path.join(DATA_PATH, action, str(sequence))
                if not os.path.exists(npy_dir):
                    os.makedirs(npy_dir)
                npy_path = os.path.join(npy_dir, str(frame_num))
                np.save(npy_path, keypoints)

                # Break gracefully
                if cv2.waitKey(10) & 0xFF == ord('q'):
                    stop = True
                    break

cap.release()
cv2.destroyAllWindows()

# **Yo bhanda tala ko garna pardaina ahile**

**Preprocess Data and Create Labels and Features**

In [9]:
from sklearn.model_selection import train_test_split
from tensorflow.keras.utils import to_categorical

In [10]:
label_map = {label:num for num, label in enumerate(actions)}
label_map

{'always': 0,
 'ask': 1,
 'bathroom': 2,
 'bird': 3,
 'black': 4,
 'blue': 5,
 'book': 6,
 'brown': 7,
 'busy': 8,
 'buy': 9,
 'candy': 10,
 'car': 11,
 'cat': 12,
 'clean': 13,
 'come': 14,
 'cook': 15,
 'deaf': 16,
 'draw': 17,
 'drink': 18,
 'eat': 19,
 'fine': 20,
 'finish': 21,
 'forget': 22,
 'give': 23,
 'go': 24,
 'good': 25,
 'green': 26,
 'happy': 27,
 'hello': 28,
 'help': 29,
 'house': 30,
 'how': 31,
 'hungry': 32,
 'i': 33,
 'icecream': 34,
 'know': 35,
 'learn': 36,
 'like': 37,
 'love_it': 38,
 'man': 39,
 'milk': 40,
 'more': 41,
 'name': 42,
 'never': 43,
 'no': 44,
 'not': 45,
 'pay_attention': 46,
 'play': 47,
 'please': 48,
 'red': 49,
 'right': 50,
 'room': 51,
 'sad': 52,
 'same': 53,
 'say': 54,
 'see': 55,
 'shhh': 56,
 'sleep': 57,
 'sorry': 58,
 'test': 59,
 'text': 60,
 'thankyou': 61,
 'time': 62,
 'today': 63,
 'tomorrow': 64,
 'understand': 65,
 'walk': 66,
 'want': 67,
 'water': 68,
 'what': 69,
 'where': 70,
 'white': 71,
 'who': 72,
 'woman': 73,
 'wor

In [11]:
sequences, labels = [], []
for action in actions:
    print(f"Processing action: {action}")
    for sequence in np.array(os.listdir(os.path.join(DATA_PATH, action))).astype(int):
        window = []
        for frame_num in range(sequence_length):
            res = np.load(os.path.join(DATA_PATH, action, str(sequence), "{}.npy".format(frame_num)))
            window.append(res)
        sequences.append(window)
        labels.append(label_map[action])
np.array(sequences).shape

Processing action: always
Processing action: ask
Processing action: bathroom
Processing action: bird
Processing action: black
Processing action: blue
Processing action: book
Processing action: brown
Processing action: busy
Processing action: buy
Processing action: candy
Processing action: car
Processing action: cat
Processing action: clean
Processing action: come
Processing action: cook
Processing action: deaf
Processing action: draw
Processing action: drink
Processing action: eat
Processing action: fine
Processing action: finish
Processing action: forget
Processing action: give
Processing action: go
Processing action: good
Processing action: green
Processing action: happy
Processing action: hello
Processing action: help
Processing action: house
Processing action: how
Processing action: hungry
Processing action: i
Processing action: icecream
Processing action: know
Processing action: learn
Processing action: like
Processing action: love_it
Processing action: man
Processing action: milk

(1600, 30, 546)

In [17]:
np.array(sequences).shape

(80, 30, 1662)

In [13]:
np.array(labels).shape

(1600,)

In [12]:
X = np.array(sequences)

In [14]:
X.shape

(1600, 30, 546)

In [15]:
y = to_categorical(labels).astype(int)

In [16]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.05)

In [17]:
y_test.shape

(80, 80)

https://github.com/nicknochnack/ActionDetectionforSignLanguage/blob/main/Action%20Detection%20Refined.ipynb

https://youtu.be/doDUihpj6ro?si=xUUwrOa0YeQoVfqC&t=5640

#**Build and Train LSTM Neural Network**

In [18]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense
from tensorflow.keras.callbacks import TensorBoard

In [19]:
log_dir = os.path.join('Logs')
tb_callback = TensorBoard(log_dir=log_dir)

In [21]:
from tensorflow.keras.layers import Dropout, BatchNormalization

model = Sequential()
model.add(LSTM(64, return_sequences=True, activation='tanh', input_shape=(30,546)))
model.add(Dropout(0.3))
model.add(LSTM(128, return_sequences=True, activation='tanh'))
model.add(Dropout(0.3))
model.add(LSTM(64, return_sequences=False, activation='tanh'))
model.add(Dropout(0.3))
model.add(Dense(64, activation='relu'))
model.add(BatchNormalization())
model.add(Dense(32, activation='relu'))
model.add(Dense(actions.shape[0], activation='softmax'))

In [22]:
from tensorflow.keras.optimizers import Adam
optimizer = Adam(learning_rate=0.0005)
model.compile(optimizer=optimizer, loss='categorical_crossentropy', metrics=['categorical_accuracy'])

In [23]:
from tensorflow.keras.callbacks import EarlyStopping
es = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)
model.fit(X_train, y_train, validation_split=0.1, callbacks=[tb_callback, es], epochs=60)

Epoch 1/60
43/43 ━━━━━━━━━━━━━━━━━━━━ 11s 102ms/step - categorical_accuracy: 0.0175 - loss: 4.4477 - val_categorical_accuracy: 0.0066 - val_loss: 4.3728
Epoch 2/60
43/43 ━━━━━━━━━━━━━━━━━━━━ 3s 80ms/step - categorical_accuracy: 0.0212 - loss: 4.4361 - val_categorical_accuracy: 0.0395 - val_loss: 4.3381
Epoch 3/60
43/43 ━━━━━━━━━━━━━━━━━━━━ 4s 84ms/step - categorical_accuracy: 0.0232 - loss: 4.2549 - val_categorical_accuracy: 0.0263 - val_loss: 4.2376
Epoch 4/60
43/43 ━━━━━━━━━━━━━━━━━━━━ 3s 80ms/step - categorical_accuracy: 0.0534 - loss: 4.0444 - val_categorical_accuracy: 0.0592 - val_loss: 4.0834
Epoch 5/60
43/43 ━━━━━━━━━━━━━━━━━━━━ 4s 82ms/step - categorical_accuracy: 0.0682 - loss: 3.8253 - val_categorical_accuracy: 0.0789 - val_loss: 3.8119
Epoch 6/60
43/43 ━━━━━━━━━━━━━━━━━━━━ 4s 82ms/step - categorical_accuracy: 0.1087 - loss: 3.5519 - val_categorical_accuracy: 0.1316 - val_loss: 3.6050
Epoch 7/60
43/43 ━━━━━━━━━━━━━━━━━━━━ 4s 84ms/step - categorical_accuracy: 0.1447 - loss: 3.

In [24]:
model.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm_3 (LSTM)                   │ (None, 30, 64)         │       156,416 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_3 (Dropout)             │ (None, 30, 64)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_4 (LSTM)                   │ (None, 30, 128)        │        98,816 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_4 (Dropout)             │ (None, 30, 128)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_5 (LSTM)                   │ (None, 64)             │        49,408 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_5 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 64)             │         4,160 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 64)             │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 80)             │         2,640 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 941,074 (3.59 MB)

 Trainable params: 313,648 (1.20 MB)

 Non-trainable params: 128 (512.00 B)

 Optimizer params: 627,298 (2.39 MB)

#**Make Predictions**

In [25]:
res = model.predict(X_test)
print(actions[np.argmax(res[2])])


3/3 ━━━━━━━━━━━━━━━━━━━━ 1s 183ms/step
car


In [26]:
actions[np.argmax(res)]

IndexError: index 5916 is out of bounds for axis 0 with size 80

In [27]:
actions[np.argmax(y_test[2])]

'car'

#**Save Weights and Model**


In [28]:
model.save('/home/sanjib/Desktop/backend/src/word/collective2.h5')

#**Evaluation using Confusion Matrix and Accuracy**

In [29]:
from sklearn.metrics import multilabel_confusion_matrix, accuracy_score

In [30]:
yhat = model.predict(X_test)

3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 30ms/step


In [31]:
ytrue = np.argmax(y_test, axis=1).tolist()
yhat = np.argmax(yhat, axis=1).tolist()

In [32]:
multilabel_confusion_matrix(ytrue, yhat)

/home/sanjib/Desktop/backend/mp-env/lib/python3.12/site-packages/sklearn/metrics/_classification.py:98: UserWarning: The number of unique classes is greater than 50% of the number of samples.
  type_true = type_of_target(y_true, input_name="y_true")
/home/sanjib/Desktop/backend/mp-env/lib/python3.12/site-packages/sklearn/metrics/_classification.py:99: UserWarning: The number of unique classes is greater than 50% of the number of samples.
  type_pred = type_of_target(y_pred, input_name="y_pred")
/home/sanjib/Desktop/backend/mp-env/lib/python3.12/site-packages/sklearn/utils/multiclass.py:79: UserWarning: The number of unique classes is greater than 50% of the number of samples.
  ys_types = set(type_of_target(x) for x in ys)
/home/sanjib/Desktop/backend/mp-env/lib/python3.12/site-packages/sklearn/utils/multiclass.py:79: UserWarning: The number of unique classes is greater than 50% of the number of samples.
  ys_types = set(type_of_target(x) for x in ys)


array([[[79,  0],
        [ 0,  1]],

       [[79,  0],
        [ 1,  0]],

       [[79,  1],
        [ 0,  0]],

       [[79,  0],
        [ 0,  1]],

       [[79,  0],
        [ 0,  1]],

       [[79,  0],
        [ 0,  1]],

       [[78,  0],
        [ 0,  2]],

       [[79,  0],
        [ 0,  1]],

       [[79,  0],
        [ 0,  1]],

       [[79,  0],
        [ 0,  1]],

       [[78,  0],
        [ 0,  2]],

       [[79,  0],
        [ 0,  1]],

       [[79,  0],
        [ 0,  1]],

       [[79,  0],
        [ 0,  1]],

       [[79,  0],
        [ 0,  1]],

       [[79,  0],
        [ 0,  1]],

       [[78,  1],
        [ 1,  0]],

       [[78,  0],
        [ 0,  2]],

       [[79,  0],
        [ 0,  1]],

       [[78,  0],
        [ 0,  2]],

       [[79,  0],
        [ 0,  1]],

       [[79,  0],
        [ 0,  1]],

       [[78,  0],
        [ 1,  1]],

       [[78,  0],
        [ 0,  2]],

       [[79,  0],
        [ 0,  1]],

       [[78,  0],
        [ 0,  2]],

       [[78,

In [33]:
accuracy_score(ytrue, yhat)

/home/sanjib/Desktop/backend/mp-env/lib/python3.12/site-packages/sklearn/metrics/_classification.py:98: UserWarning: The number of unique classes is greater than 50% of the number of samples.
  type_true = type_of_target(y_true, input_name="y_true")
/home/sanjib/Desktop/backend/mp-env/lib/python3.12/site-packages/sklearn/metrics/_classification.py:99: UserWarning: The number of unique classes is greater than 50% of the number of samples.
  type_pred = type_of_target(y_pred, input_name="y_pred")


0.95

#**Test in Real Time**

In [34]:
from scipy import stats

In [51]:
colors = [(245,117,16), (117,245,16), (16,117,245)]
def prob_viz(res, actions, input_frame, colors):
    output_frame = input_frame.copy()
    for num, prob in enumerate(res):
        cv2.rectangle(output_frame, (0,60+num*40), (int(prob*100), 90+num*40), colors[num], -1)
        cv2.putText(output_frame, actions[num], (0, 85+num*40), cv2.FONT_HERSHEY_SIMPLEX, 1, (255,255,255), 2, cv2.LINE_AA)

    return output_frame

In [52]:
plt.figure(figsize=(18,18))
plt.imshow(prob_viz(res, actions, image, colors))

TypeError: only length-1 arrays can be converted to Python scalars

<Figure size 1800x1800 with 0 Axes>

In [ ]:
# 1. New detection variables
sequence = []
sentence = []
predictions = []
threshold = 0.5

cap = cv2.VideoCapture(0)
# Set mediapipe model
with mp_holistic.Holistic(min_detection_confidence=0.5, min_tracking_confidence=0.5) as holistic:
    while cap.isOpened():

        # Read feed
        ret, frame = cap.read()

        # Make detections
        image, results = mediapipe_detection(frame, holistic)
        print(results)

        # Draw landmarks
        draw_styled_landmarks(image, results)

        # 2. Prediction logic
        keypoints = extract_keypoints(results)
        sequence.append(keypoints)
        sequence = sequence[-30:]

        if len(sequence) == 30:
            res = model.predict(np.expand_dims(sequence, axis=0))[0]
            print(actions[np.argmax(res)])
            predictions.append(np.argmax(res))


        #3. Viz logic
            if np.unique(predictions[-10:])[0]==np.argmax(res):
                if res[np.argmax(res)] > threshold:

                    if len(sentence) > 0:
                        if actions[np.argmax(res)] != sentence[-1]:
                            sentence.append(actions[np.argmax(res)])
                    else:
                        sentence.append(actions[np.argmax(res)])

            if len(sentence) > 5:
                sentence = sentence[-5:]

            # Viz probabilities
            image = prob_viz(res, actions, image, colors)

        cv2.rectangle(image, (0,0), (640, 40), (245, 117, 16), -1)
        cv2.putText(image, ' '.join(sentence), (3,30),
                       cv2.FONT_HERSHEY_SIMPLEX, 1, (255, 255, 255), 2, cv2.LINE_AA)

        # Show to screen
        cv2.imshow('OpenCV Feed', image)

        # Break gracefully
        if cv2.waitKey(10) & 0xFF == ord('q'):
            break
    cap.release()
    cv2.destroyAllWindows()

<class 'mediapipe.python.solution_base.SolutionOutputs'>
<class 'mediapipe.python.solution_base.SolutionOutputs'>
<class 'mediapipe.python.solution_base.SolutionOutputs'>
<class 'mediapipe.python.solution_base.SolutionOutputs'>
<class 'mediapipe.python.solution_base.SolutionOutputs'>
<class 'mediapipe.python.solution_base.SolutionOutputs'>
<class 'mediapipe.python.solution_base.SolutionOutputs'>
<class 'mediapipe.python.solution_base.SolutionOutputs'>
<class 'mediapipe.python.solution_base.SolutionOutputs'>
<class 'mediapipe.python.solution_base.SolutionOutputs'>
<class 'mediapipe.python.solution_base.SolutionOutputs'>
<class 'mediapipe.python.solution_base.SolutionOutputs'>
<class 'mediapipe.python.solution_base.SolutionOutputs'>
<class 'mediapipe.python.solution_base.SolutionOutputs'>
<class 'mediapipe.python.solution_base.SolutionOutputs'>
<class 'mediapipe.python.solution_base.SolutionOutputs'>
<class 'mediapipe.python.solution_base.SolutionOutputs'>
<class 'mediapipe.python.soluti

IndexError: list index out of range

: 